# P5-6 · GĐ 6 · Đánh giá và rectified flow

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
def frechet(a, b):                         # a, b: (N, d) đặc trưng của hai tập
    ma, mb = a.mean(0), b.mean(0)
    ca, cb = torch.cov(a.T), torch.cov(b.T)
    tr_sqrt = torch.linalg.eigvals(ca @ cb).real.clamp(min=0).sqrt().sum()   # Tr((Σa Σb)^½)
    return ((ma - mb) ** 2).sum() + ca.trace() + cb.trace() - 2 * tr_sqrt

## Bài tập 4: Flow matching trên điểm 2D {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

def make_spiral(n):
    th = torch.sqrt(torch.rand(n)) * 3 * math.pi
    return torch.stack([th * torch.cos(th), th * torch.sin(th)], 1) / (3 * math.pi) + 0.02 * torch.randn(n, 2)

class VNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, t[:, None]], 1))

data, model = make_spiral(10000), VNet()
opt = torch.optim.Adam(model.parameters(), 1e-3)
for step in range(4000):
    x0 = data[torch.randint(0, len(data), (512,))]
    eps, t = torch.randn_like(x0), torch.rand(512)
    xt = (1 - t[:, None]) * x0 + t[:, None] * eps
    loss = F.mse_loss(model(xt, t), eps - x0)
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def sample(n=2000, steps=10):
    x, dt = torch.randn(n, 2), 1.0 / steps
    for i in range(steps, 0, -1):
        x = x - dt * model(x, torch.full((n,), i * dt))
    return x

pts = sample()
print("khoảng cách TB tới dữ liệu:", torch.cdist(pts, data[:5000]).min(1).values.mean().item())

khoảng cách TB tới dữ liệu: 0.03564347326755524


## Bài tập 5: FID trên đặc trưng nhiều chiều {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from scipy.linalg import sqrtm
rng = np.random.default_rng(0)
d, n = 16, 2000
L = rng.normal(0, 0.3, (d, d))
that = rng.multivariate_normal(np.zeros(d), np.eye(d) + L @ L.T, n)
tot = rng.multivariate_normal(np.zeros(d), np.eye(d) + L @ L.T, n) * 1.03
lech = that + 0.5 + rng.normal(0, 0.05, (n, d))
sup_do = that.mean(0) + 0.3 * (rng.multivariate_normal(np.zeros(d), np.eye(d) + L @ L.T, n))

def fid(A, B):
    # TODO
    return 0.0

In [ ]:
def fid(A, B):
    mu1, mu2 = A.mean(0), B.mean(0)
    S1, S2 = np.cov(A, rowvar=False), np.cov(B, rowvar=False)
    cs = sqrtm(S1 @ S2).real
    return float(((mu1 - mu2) ** 2).sum() + np.trace(S1 + S2 - 2 * cs))

kq = {ten: fid(that, B) for ten, B in [("tốt", tot), ("lệch", lech), ("sụp đổ", sup_do)]}
for ten, v in sorted(kq.items(), key=lambda p: p[1]):
    print(f"{ten:7s}: FID {v:.2f}")
print("FID thật với chính nó (hai nửa):", round(fid(that[:1000], that[1000:]), 2))

tốt    : FID 0.25
lệch   : FID 3.99
sụp đổ : FID 19.31
FID thật với chính nó (hai nửa): 0.29


## Bài tập 6: Phân tích khảo sát người dùng so hai model {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
nguoi = rng.integers(0, 8, 120)                     # 8 người chấm
cau_thua = {3: 3, 6: 2}                             # người 3 sai 3 câu kiểm tra, người 6 sai 2
ket_qua = rng.choice(["A", "B", "hòa"], 120, p=[0.52, 0.33, 0.15])
ket_qua[nguoi == 3] = rng.choice(["A", "B"], (nguoi == 3).sum())     # người cẩu thả: chọn bừa
# TODO: loại người sai >= 2 câu kiểm tra; diem (1 / 0,5 / 0); win_rate; ci (bootstrap 5000 lần)

In [ ]:
giu = ~np.isin(nguoi, [k for k, v in cau_thua.items() if v >= 2])
kq = ket_qua[giu]
diem = np.select([kq == "A", kq == "hòa"], [1.0, 0.5], 0.0)
win_rate = diem.mean()
rb = np.random.default_rng(1)
bs = [diem[rb.integers(0, len(diem), len(diem))].mean() for _ in range(5000)]
ci = np.percentile(bs, [2.5, 97.5])
print(f"giữ {giu.sum()}/120 lượt | A thắng {win_rate:.1%}, khoảng tin cậy 95% [{ci[0]:.1%}, {ci[1]:.1%}]")

giữ 87/120 lượt | A thắng 59.2%, khoảng tin cậy 95% [50.0%, 67.8%]
